# CounterStrike-1K Dataset Viewer

Index-based browser for the CS2-WM dataset. Reads a frozen manifest and
lets you flip through clips, inspect actions, and (default) view the
debug overlay rendered on demand.

**Two modes:**
- `from_local(...)` — reads from a directory produced by `cs2_train/src/download.py`.
- `from_manifest_file(...)` — reads a manifest JSON; clips are fetched lazily from S3 to a cache.

**Debug overlay** (default ON) re-renders the keyboard / mouse / weapon HUD
on top of each clip the first time it is requested, then caches the result.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, json
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

from cs2_train.baselines.diamond.dataset import DatasetViewer

## Configure source

Pick **one** of the two construction blocks below.
Set `DEBUG_OVERLAY=False` to view the raw clips without the HUD.

In [ ]:
DEBUG_OVERLAY = True   # default ON — flip to False for raw clips

# --- Mode A: local directory (downloaded via cs2_train/src/download.py) ---
# v = DatasetViewer.from_local(
#     '/opt/dlami/nvme/cs2-data',
#     manifest_name='manifest.json',
#     debug_overlay=DEBUG_OVERLAY,
# )

# --- Mode B: frozen manifest, lazy fetch from S3 ---
v = DatasetViewer.from_manifest_file(
    REPO_ROOT / 'cs2_train' / 'data' / 'manifests' / 'manifest_100h.json',
    cache_dir='/tmp/cs2_viewer_cache',
    debug_overlay=DEBUG_OVERLAY,
)

summary = v.summary()
for k, val in summary.items():
    print(f'  {k:>10}: {val}')

## Single-clip view

Pull one sample by index, render its overlay, show the video + action plot.

In [ ]:
import IPython.display as ipd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def show(idx: int, max_action_rows: int = 20):
    s = v[idx]
    print(f'idx={s.index}')
    for k, val in s.metadata.items():
        print(f'  {k:>11}: {val}')
    ipd.display(ipd.Video(str(s.video_path), embed=True, width=640))
    df = s.actions_df.head(max_action_rows)
    ipd.display(df)
    fig, ax = plt.subplots(2, 1, figsize=(11, 4), sharex=True)
    full_df = s.actions_df
    btns = ['FORWARD','BACK','LEFT','RIGHT','JUMP','DUCK','WALK','FIRE',
            'RIGHTCLICK','RELOAD','INSPECT','USE']
    btns = [c for c in btns if c in full_df.columns]
    grid = full_df[btns].astype(int).T.to_numpy()
    ax[0].imshow(grid, aspect='auto', cmap='Reds', interpolation='nearest')
    ax[0].set_yticks(range(len(btns)))
    ax[0].set_yticklabels(btns, fontsize=8)
    ax[0].set_title(f'buttons over {len(full_df)} frames')
    if 'delta_pitch' in full_df.columns and 'delta_yaw' in full_df.columns:
        ax[1].plot(full_df['delta_yaw'].to_numpy(), label='delta_yaw', lw=0.6)
        ax[1].plot(full_df['delta_pitch'].to_numpy(), label='delta_pitch', lw=0.6)
        ax[1].set_ylabel('mouse delta')
        ax[1].legend(fontsize=8, loc='upper right')
    ax[1].set_xlabel('frame')
    plt.tight_layout()
    plt.show()
    return s

_ = show(0)

## Interactive browser

Slide through clips. Toggle the debug overlay live.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

out = widgets.Output()

idx_slider = widgets.IntSlider(
    min=0, max=len(v) - 1, step=1, value=0, description='clip',
    continuous_update=False,
    layout=widgets.Layout(width='80%'),
)
search_box = widgets.Text(
    placeholder='filter by player_name / match_id (substring) — submit to apply',
    description='filter:', layout=widgets.Layout(width='80%'),
)
debug_toggle = widgets.Checkbox(value=DEBUG_OVERLAY, description='debug overlay')

filtered_idx = list(range(len(v)))

def apply_filter(_=None):
    global filtered_idx
    q = (search_box.value or '').strip().lower()
    if not q:
        filtered_idx = list(range(len(v)))
    else:
        filtered_idx = [
            i for i, c in enumerate(v.manifest)
            if q in str(c.get('player_name', '')).lower()
            or q in str(c.get('source_demo_id', c.get('match_id', ''))).lower()
            or q in str(c.get('player_id', '')).lower()
        ]
    if not filtered_idx:
        filtered_idx = [0]
    idx_slider.max = len(filtered_idx) - 1
    idx_slider.value = 0
    render(0)

def render(slider_val=None):
    v.debug_overlay = debug_toggle.value
    real_idx = filtered_idx[idx_slider.value]
    with out:
        clear_output(wait=True)
        show(real_idx)

idx_slider.observe(lambda ch: render() if ch['name'] == 'value' else None,
                   names='value')
debug_toggle.observe(lambda ch: render() if ch['name'] == 'value' else None,
                     names='value')
search_box.on_submit(apply_filter)

display(widgets.VBox([search_box, debug_toggle, idx_slider, out]))
render()

## Quick speed sanity-check

Smoke test: pull 50 clips, time the per-clip access. **Not** the proper
benchmark — for that, see `cs2_train/scripts/bench/run_bench.py` and the
article at `learning/dataloader-2026/`.

In [ ]:
import time, random
rng = random.Random(0)
ids = rng.sample(range(len(v)), min(50, len(v)))
t0 = time.time()
ok = bad = 0
for i in ids:
    try:
        _ = v[i]
        ok += 1
    except Exception as e:
        bad += 1
        print(f'  bad idx={i}: {e!r}')
wall = time.time() - t0
print(f'ok={ok}, bad={bad}, wall={wall:.2f}s, '
      f'{ok/max(wall,1e-6):.1f} clips/s '
      f'(includes overlay render the first time per clip)')